# FeTA Preprocessing — Official SSL3D / nnU-Net

**Reproducible record of the validated FeTA pipeline** (80 T2w MRIs, 48 train / 16 validation / 16 test).

This notebook calls the scripts in `scripts/` and the official `petermcgor/nnunetv2:1.1.0-nnssl` Docker image. **It is not the obsolete FOMO preprocessing pipeline.**

> **FeTA was already preprocessed and validated.** Start with **Step 0 — Check existing outputs**; rerun conversion or preprocessing *only if rebuilding the dataset*. All potentially expensive cells are protected by `RUN_PIPELINE = False` by default.

Run on **Windows, with Docker Desktop running**, from a local Python notebook kernel. Execute the notebook from the repository or `notebooks/` directory.

## Setup — paths, dataset and Docker

In [1]:
from pathlib import Path
import json
import subprocess
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

IMAGE = "petermcgor/nnunetv2:1.1.0-nnssl"
RAW_DIR = ROOT / "data" / "nnUNet_raw"
PREPROCESSED_DIR = ROOT / "data" / "nnUNet_preprocessed"
RESULTS_DIR = ROOT / "data" / "nnUNet_results"
SPLITS_PATH = ROOT / "data" / "splits" / "feta_splits.json"
DATASET_RAW = RAW_DIR / "Dataset101_FeTA"
DATASET_PP = PREPROCESSED_DIR / "Dataset101_FeTA"
TRAIN_DIR = DATASET_PP / "Spacing__1.00_1.00_1.00___Norm__Z"
VAL_DIR = DATASET_PP / "SSL3D_val"
TEST_DIR = DATASET_PP / "SSL3D_test"

# Keep False if you only want to inspect already-preprocessed data.
RUN_PIPELINE = False

assert SPLITS_PATH.is_file(), f"Missing splits: {SPLITS_PATH}"
print("Project root:", ROOT)
print("Docker image:", IMAGE)
print("Existing raw dataset:", DATASET_RAW.is_dir())
print("Existing preprocessed dataset:", DATASET_PP.is_dir())
print("RUN_PIPELINE:", RUN_PIPELINE)

Project root: C:\Users\yasmi\-brain-mri-foundation-models
Docker image: petermcgor/nnunetv2:1.1.0-nnssl
Existing raw dataset: True
Existing preprocessed dataset: True
RUN_PIPELINE: False


## Step 0 — Check existing preprocessing (safe to run anytime)

Expected output: **48 training, 16 validation, 16 test** MRIs, with one `.b2nd` image, one `_seg.b2nd` mask and one `.pkl` per subject.

In [2]:
expected = {"train": (TRAIN_DIR, 48), "val": (VAL_DIR, 16), "test": (TEST_DIR, 16)}
for split, (folder, n_expected) in expected.items():
    imgs = [p for p in folder.glob("*.b2nd") if not p.name.endswith("_seg.b2nd")]
    masks = list(folder.glob("*_seg.b2nd"))
    meta = list(folder.glob("*.pkl"))
    print(
        f"{split:>5}: MRI={len(imgs)}/{n_expected}, masks={len(masks)}/{n_expected}, metadata={len(meta)}/{n_expected}"
    )

print("Fingerprint:", (DATASET_PP / "dataset_fingerprint.json").exists())
print("nnU-Net plans:", (DATASET_PP / "nnUNetPlans.json").exists())
print(
    "SSL3D plan files:", [p.name for p in DATASET_PP.glob("ptPlans__SSL3D_FeTA*.json")]
)

train: MRI=48/48, masks=48/48, metadata=48/48
  val: MRI=16/16, masks=16/16, metadata=16/16
 test: MRI=16/16, masks=16/16, metadata=16/16
Fingerprint: True
nnU-Net plans: True
SSL3D plan files: ['ptPlans__SSL3D_FeTA____Spacing__1.00_1.00_1.00___Norm__Z.json']


## Step 1 — Convert FeTA into nnU-Net raw format

Inputs: original `data/FeTA/feta_2.4/` and frozen `data/splits/feta_splits.json`. The conversion script preserves MRI data and converts integer-valued segmentation masks to `uint8`.

Expected: `imagesTr/labelsTr` **48**, `imagesVal/labelsVal` **16**, `imagesTs/labelsTs` **16**. This step is **optional** if `Dataset101_FeTA` already exists.

In [3]:
if RUN_PIPELINE:
    subprocess.run(
        [sys.executable, str(ROOT / "scripts" / "convert_feta_to_nnunet.py")],
        cwd=ROOT,
        check=True,
    )
else:
    print("SKIPPED — set RUN_PIPELINE=True only if intentionally rebuilding FeTA.")

SKIPPED — set RUN_PIPELINE=True only if intentionally rebuilding FeTA.


In [4]:
for folder in (
    "imagesTr",
    "labelsTr",
    "imagesVal",
    "labelsVal",
    "imagesTs",
    "labelsTs",
):
    print(folder, len(list((DATASET_RAW / folder).glob("*.nii.gz"))))
print("dataset.json:", (DATASET_RAW / "dataset.json").exists())

imagesTr 48
labelsTr 48
imagesVal 16
labelsVal 16
imagesTs 16
labelsTs 16
dataset.json: True


## Step 2 — Fingerprint + nnU-Net planning (training only)

`--no_pp` extracts the fingerprint and creates standard nnU-Net plans **without preprocessing yet**. Only 48 train subjects are in `imagesTr`; validation and test are kept separate.

Output: `dataset_fingerprint.json` and `nnUNetPlans.json`.

In [5]:
def docker_mount(local_path, target, readonly=False):
    local_path = Path(local_path)
    return [
        "-v",
        f"{local_path.resolve().as_posix()}:{target}" + (":ro" if readonly else ""),
    ]


RAW_BIND = "/opt/nnunet_resources/nnUNet_raw"
PP_BIND = "/opt/nnunet_resources/nnUNet_preprocessed"
RESULTS_BIND = "/opt/nnunet_resources/nnUNet_results"

if RUN_PIPELINE:
    PREPROCESSED_DIR.mkdir(parents=True, exist_ok=True)
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    cmd = ["docker", "run", "--rm"]
    cmd += docker_mount(RAW_DIR, RAW_BIND)
    cmd += docker_mount(PREPROCESSED_DIR, PP_BIND)
    cmd += docker_mount(RESULTS_DIR, RESULTS_BIND)
    cmd += [IMAGE, "nnUNetv2_plan_and_preprocess", "-d", "101", "--no_pp", "-np", "2"]
    subprocess.run(cmd, cwd=ROOT, check=True)
else:
    print("SKIPPED — existing fingerprint and plans can be inspected below.")

SKIPPED — existing fingerprint and plans can be inspected below.


In [6]:
plans_path = DATASET_PP / "nnUNetPlans.json"
if plans_path.exists():
    plans = json.loads(plans_path.read_text(encoding="utf-8"))
    for name, config in plans.get("configurations", {}).items():
        print(
            name,
            "spacing:",
            config.get("spacing"),
            "patch size:",
            config.get("patch_size"),
            "normalization:",
            config.get("normalization_schemes"),
        )
else:
    print("Plans not found — run Step 2 when rebuilding.")

2d spacing: [None, 0.5, 0.5] patch size: [224, 192] normalization: ['ZScoreNormalization']
3d_fullres spacing: [0.5, 0.5, 0.5] patch size: [128, 128, 128] normalization: ['ZScoreNormalization']


## Step 3 — SSL3D-compatible preprocessing of 48 train subjects

Adapt the standard plans using pretrained SSL3D ResEncL checkpoint, `like_pretrained`, target **1 × 1 × 1 mm**, Z-score normalization. The read-only patched `scripts/docker_fixes/like_nnssl.py` loads the CUDA-saved checkpoint on CPU.

This runs **official** `nnUNetv2_preprocess_like_nnssl` on train subjects.

In [7]:
CPU_FIX = ROOT / "scripts" / "docker_fixes" / "like_nnssl.py"
CKPT_IN_IMAGE = (
    "/opt/nnunet_resources/models/"
    "BaseMAETrainerExtendedHealth__nnsslPlans__onemmiso/fold_all/checkpoint_best.pth"
)

if RUN_PIPELINE:
    assert CPU_FIX.is_file(), f"Missing CPU fix: {CPU_FIX}"
    cmd = ["docker", "run", "--rm"]
    cmd += docker_mount(RAW_DIR, RAW_BIND)
    cmd += docker_mount(PREPROCESSED_DIR, PP_BIND)
    cmd += docker_mount(RESULTS_DIR, RESULTS_BIND)
    cmd += docker_mount(
        CPU_FIX,
        "/opt/nnunet_adaptation/nnunetv2/experiment_planning/like_nnssl.py",
        readonly=True,
    )
    cmd += [
        IMAGE,
        "nnUNetv2_preprocess_like_nnssl",
        "-d",
        "101",
        "-n",
        "SSL3D_FeTA",
        "-pc",
        CKPT_IN_IMAGE,
        "-am",
        "like_pretrained",
        "-np",
        "1",
    ]
    subprocess.run(cmd, cwd=ROOT, check=True)
else:
    print("SKIPPED — preprocessed training data already available if Step 0 passed.")

SKIPPED — preprocessed training data already available if Step 0 passed.


In [8]:
print("SSL3D plan(s):", [p.name for p in DATASET_PP.glob("ptPlans__SSL3D_FeTA*.json")])
print(
    "Train MRIs:",
    len([p for p in TRAIN_DIR.glob("*.b2nd") if not p.name.endswith("_seg.b2nd")]),
)
print("Train masks:", len(list(TRAIN_DIR.glob("*_seg.b2nd"))))
print("Train metadata:", len(list(TRAIN_DIR.glob("*.pkl"))))

SSL3D plan(s): ['ptPlans__SSL3D_FeTA____Spacing__1.00_1.00_1.00___Norm__Z.json']
Train MRIs: 48
Train masks: 48
Train metadata: 48


## Step 4 — Preprocess the 16 validation and 16 test subjects

Use `scripts/preprocess_feta_val_test.py`, which calls nnU-Net's `DefaultPreprocessor.run_case_save()` using **the same SSL3D plans derived from train**. No additional fingerprint/planning step.

In [9]:
VAL_TEST_SCRIPT = ROOT / "scripts" / "preprocess_feta_val_test.py"
if RUN_PIPELINE:
    assert VAL_TEST_SCRIPT.is_file(), f"Missing script: {VAL_TEST_SCRIPT}"
    cmd = ["docker", "run", "--rm"]
    cmd += docker_mount(RAW_DIR, RAW_BIND)
    cmd += docker_mount(PREPROCESSED_DIR, PP_BIND)
    cmd += docker_mount(ROOT / "data" / "splits", "/opt/project_splits", readonly=True)
    cmd += docker_mount(
        VAL_TEST_SCRIPT, "/tmp/preprocess_feta_val_test.py", readonly=True
    )
    cmd += [IMAGE, "python", "/tmp/preprocess_feta_val_test.py"]
    subprocess.run(cmd, cwd=ROOT, check=True)
else:
    print("SKIPPED — existing validation and test preprocessing can be checked below.")

SKIPPED — existing validation and test preprocessing can be checked below.


In [10]:
for name, folder in (("val", VAL_DIR), ("test", TEST_DIR)):
    print(
        f"{name}: MRIs={len([p for p in folder.glob('*.b2nd') if not p.name.endswith('_seg.b2nd')])}, "
        f"masks={len(list(folder.glob('*_seg.b2nd')))}, metadata={len(list(folder.glob('*.pkl')))}"
    )

val: MRIs=16, masks=16, metadata=16
test: MRIs=16, masks=16, metadata=16


## Step 5 — Automated quality control (safe to rerun)

Use the existing FeTA QA script to validate split membership, completeness, image/mask shape consistency, finite values, labels, and metadata. Expected **TRAIN 48/48 · VAL 16/16 · TEST 16/16 · TOTAL 80/80**.

Docker provides `blosc2` and the official nnU-Net dependencies.

In [11]:
CHECK_SCRIPT = ROOT / "scripts" / "check_feta_preprocessing.py"
assert CHECK_SCRIPT.is_file(), f"Missing script: {CHECK_SCRIPT}"
cmd = ["docker", "run", "--rm"]
cmd += docker_mount(PREPROCESSED_DIR, PP_BIND, readonly=True)
cmd += docker_mount(ROOT / "data" / "splits", "/opt/project_splits", readonly=True)
cmd += docker_mount(CHECK_SCRIPT, "/tmp/check_feta_preprocessing.py", readonly=True)
cmd += [IMAGE, "python", "/tmp/check_feta_preprocessing.py"]
subprocess.run(cmd, cwd=ROOT, check=True)

CompletedProcess(args=['docker', 'run', '--rm', '-v', 'C:/Users/yasmi/-brain-mri-foundation-models/data/nnUNet_preprocessed:/opt/nnunet_resources/nnUNet_preprocessed:ro', '-v', 'C:/Users/yasmi/-brain-mri-foundation-models/data/splits:/opt/project_splits:ro', '-v', 'C:/Users/yasmi/-brain-mri-foundation-models/scripts/check_feta_preprocessing.py:/tmp/check_feta_preprocessing.py:ro', 'petermcgor/nnunetv2:1.1.0-nnssl', 'python', '/tmp/check_feta_preprocessing.py'], returncode=0)

## Step 6 — Visual quality control (optional)

Generate selected MRI slices and segmentation overlays using the existing script. Outputs are generated in `data/qa_feta_ssl3d/` (**gitignored**). Use `--all` to generate visualizations for all 80 subjects.

In [12]:
VIS_SCRIPT = ROOT / "scripts" / "visualize_feta_preprocessing.py"
QA_DIR = ROOT / "data" / "qa_feta_ssl3d"
QA_DIR.mkdir(parents=True, exist_ok=True)

assert VIS_SCRIPT.is_file(), f"Missing script: {VIS_SCRIPT}"
cmd = ["docker", "run", "--rm"]
cmd += docker_mount(PREPROCESSED_DIR, PP_BIND, readonly=True)
cmd += docker_mount(ROOT / "data" / "splits", "/opt/project_splits", readonly=True)
cmd += docker_mount(QA_DIR, "/opt/qa_output")
cmd += docker_mount(VIS_SCRIPT, "/tmp/visualize_feta_preprocessing.py", readonly=True)
cmd += [IMAGE, "python", "/tmp/visualize_feta_preprocessing.py"]
# To visualize all 80 subjects: cmd.append("--all")
subprocess.run(cmd, cwd=ROOT, check=True)
print("QA output directory:", QA_DIR)

QA output directory: C:\Users\yasmi\-brain-mri-foundation-models\data\qa_feta_ssl3d


## Final handoff

Once Step 5 passes, the validated FeTA data are ready for frozen-encoder feature extraction and segmentation-head training.

- **Code and splits** → GitHub (`scripts/`, `notebooks/`, `data/splits/feta_splits.json`).
- **Preprocessed volumes and plans** → transferred separately to Izar (`data/nnUNet_preprocessed/Dataset101_FeTA/`).
- **Raw original-resolution images and masks** → keep accessible for final native-space evaluation.
- **No pretraining/fine-tuning takes place here.** The trainable segmentation heads are implemented separately.

All steps are repeatable in order, but rebuilding is unnecessary if the output and QA checks already pass.